# Belgium Hourly Weather Crawler
Crawl hourly weather data for Belgian regions from **Dec 1, 2024 to Mar 25, 2026**.  
Uses the **Open-Meteo Archive API** (historical) + **Forecast API** (recent days).  
Requests are chunked by year to avoid timeouts and rate limits.  
Supports **resuming** — already-complete sites are skipped automatically.


In [11]:
import requests
import csv
import time
from collections import Counter
from datetime import datetime, date

import pandas as pd


In [ ]:
#Loading sites data
sites_columns=['site_id', 'site_nr', 'longtitude', 'latitude', 'name', 'domain', 'road_nbr','district_nbr', 'municipality', 'interval', 'date_installed']
df_sites = pd.read_csv("D:/KUL MSDS 1/mda_project/data/sites.csv", header=1, names=sites_columns)
df_sites.head()

,site_id,site_nr,longtitude,latitude,name,domain,road_nbr,district_nbr,municipality,interval,date_installed
0,2,100052862,4.47169,51.27512,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,2019-08-22
1,3,100052863,4.47222,51.27503,Brasschaat 1,Vlaamse Overheid A. Wegen enVerkeer,N0010001,AWV123,Brasschaat,15,2019-08-22
2,4,100052864,5.19011,51.16023,Balen 1,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22
3,5,100052865,5.19003,51.16018,Balen 2,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22
4,6,100052866,3.70298,51.10894,Evergem 2,Vlaamse Overheid A. Wegen enVerkeer,N4560001,AWV413,Evergem,15,2019-08-22


In [13]:
SITES = (
    df_sites[["site_id", "latitude", "longtitude"]]
    .drop_duplicates("site_id")
    .rename(columns={"longtitude": "longitude"})
    .sort_values("site_id")
    .apply(lambda r: {
        "site_id":    int(r["site_id"]),
        "latitude":  float(r["latitude"]),   # cast here once
        "longitude": float(r["longitude"]),
    }, axis=1)
    .tolist()
)

In [14]:
print(SITES)

[{'site_id': 2, 'latitude': 51.27512, 'longitude': 4.47169}, {'site_id': 3, 'latitude': 51.27503, 'longitude': 4.47222}, {'site_id': 4, 'latitude': 51.16023, 'longitude': 5.19011}, {'site_id': 5, 'latitude': 51.16018, 'longitude': 5.19003}, {'site_id': 6, 'latitude': 51.10894, 'longitude': 3.70298}, {'site_id': 7, 'latitude': 51.1089, 'longitude': 3.70317}, {'site_id': 8, 'latitude': 51.06759, 'longitude': 4.68823}, {'site_id': 9, 'latitude': 51.06904, 'longitude': 4.69524}, {'site_id': 10, 'latitude': 50.93549, 'longitude': 4.01571}, {'site_id': 11, 'latitude': 50.93385, 'longitude': 4.01647}, {'site_id': 12, 'latitude': 50.81896, 'longitude': 4.47803}, {'site_id': 13, 'latitude': 51.03018, 'longitude': 3.70146}, {'site_id': 14, 'latitude': 50.97172, 'longitude': 5.48927}, {'site_id': 15, 'latitude': 51.21371, 'longitude': 2.92743}, {'site_id': 16, 'latitude': 50.83332, 'longitude': 3.27777}, {'site_id': 17, 'latitude': 50.83339, 'longitude': 3.27773}, {'site_id': 18, 'latitude': 51.2

## Configuration

In [ ]:

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

START_DATE = "2024-12-01"
END_DATE   = "2026-03-25"
ARCHIVE_CUTOFF = date(2026, 3, 15)
HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "wind_speed_10m",
    "wind_direction_10m",
    "pressure_msl",
    "cloud_cover",
]

OUTPUT_FILE = "D:/KUL MSDS 1/mda_project/data/extra/site_id_weather_output_NEW.csv"

print(f"Regions  : {len(SITES)}")
print(f"Variables: {len(HOURLY_VARIABLES)}")
print(f"Period   : {START_DATE} → {END_DATE}")
print(f"Output   : {OUTPUT_FILE}")


Regions  : 150
Variables: 9
Period   : 2019-08-01 → 2026-03-25
Output   : D:/KUL MSDS 1/mda_project/data/extra/site_id_weather_output.csv


## Helper functions

In [16]:
def fetch_weather(url: str, lat: float, lon: float,
                  start: str, end: str, max_retries: int = 5) -> dict:
    """GET one chunk with exponential-style backoff on 429 rate limit."""
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "start_date": start,
        "end_date":   end,
        "hourly":     ",".join(HOURLY_VARIABLES),
        "timezone":   "Europe/Brussels",
    }
    for attempt in range(max_retries):
        resp = requests.get(url, params=params, timeout=60)
        if resp.status_code == 429:
            wait = 30 * (attempt + 1)   # 30s → 60s → 90s → 120s → 150s
            print(f"[429 rate limit] waiting {wait}s (attempt {attempt+1}/{max_retries})...",
                  end=" ", flush=True)
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()
    raise RuntimeError("Max retries exceeded (429)")


In [17]:
def build_year_chunks(start, end, cutoff) -> list[tuple]:
    """Split the full date range into yearly archive/forecast chunks."""
    # Accept both date objects and "YYYY-MM-DD" strings
    if isinstance(start,  str): start  = date.fromisoformat(start)
    if isinstance(end,    str): end    = date.fromisoformat(end)
    if isinstance(cutoff, str): cutoff = date.fromisoformat(cutoff)

    chunks  = []
    current = start
    while current <= end:
        chunk_end = min(date(current.year, 12, 31), end)
        if current <= cutoff:
            api_end = min(chunk_end, cutoff)
            chunks.append(("archive", current, api_end))
            if chunk_end > cutoff:
                chunks.append(("forecast", cutoff, chunk_end))
        else:
            chunks.append(("forecast", current, chunk_end))
        current = date(current.year + 1, 1, 1)
    return chunks

In [18]:
def expected_hourly_records(start: str, end: str) -> int:
    """
    Approximate expected number of hourly rows for one site.
    Used only to decide whether an existing site is complete enough to skip.
    """
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end) + pd.Timedelta(days=1)  # include end date
    return int((end_dt - start_dt).total_seconds() // 3600)


EXPECTED_ROWS_PER_SITE = expected_hourly_records(START_DATE, END_DATE)
COMPLETION_THRESHOLD = int(EXPECTED_ROWS_PER_SITE * 0.95)  # allow small API gaps


def load_existing(path: str) -> tuple[list, set]:
    """
    Load CSV if it exists.

    Because this version only crawls from START_DATE onward, existing rows before
    START_DATE are ignored. A site is considered complete when it has at least
    95% of the expected hourly records for the selected date range.
    """
    rows = []
    complete_sites = set()

    try:
        with open(path, "r", encoding="utf-8") as f:
            rows = list(csv.DictReader(f))

        # Keep only rows from START_DATE onward, so old 2019-2024 rows do not stay in memory
        rows = [
            r for r in rows
            if pd.to_datetime(r["datetime"]).date() >= date.fromisoformat(START_DATE)
        ]

        counts = Counter(str(r["site_id"]) for r in rows)
        complete_sites = {
            int(site_id)
            for site_id, n in counts.items()
            if n >= COMPLETION_THRESHOLD
        }

        print(
            f"Loaded {len(rows):,} existing rows from {START_DATE} onward | "
            f"{len(complete_sites)} complete sites "
            f"(threshold: {COMPLETION_THRESHOLD:,} rows/site)"
        )

        if complete_sites:
            print("  Skipping:", ", ".join(map(str, sorted(complete_sites))))

    except FileNotFoundError:
        print("No existing file found — starting fresh.")

    return rows, complete_sites


## Preview date chunks

In [19]:
chunks = build_year_chunks(START_DATE, END_DATE, ARCHIVE_CUTOFF)
print(f"{len(chunks)} chunks:\n")
for api, s, e in chunks:
    print(f"  [{api:>8}]  {s}  →  {e}")


9 chunks:

  [ archive]  2019-08-01  →  2019-12-31
  [ archive]  2020-01-01  →  2020-12-31
  [ archive]  2021-01-01  →  2021-12-31
  [ archive]  2022-01-01  →  2022-12-31
  [ archive]  2023-01-01  →  2023-12-31
  [ archive]  2024-01-01  →  2024-12-31
  [ archive]  2025-01-01  →  2025-12-31
  [ archive]  2026-01-01  →  2026-03-15
  [forecast]  2026-03-15  →  2026-03-25


## Crawl all regions

In [ ]:
import random
from datetime import datetime

# Tunable rate-limit settings 
DELAY_BETWEEN_CHUNKS = 3.0   # seconds between every API call
DELAY_BETWEEN_SITES  = 8.0   # extra pause after finishing each site


def fetch_weather(url: str, lat: float, lon: float,
                  start: str, end: str, max_retries: int = 7,
                  status_label: str = "") -> dict:
    """
    GET one chunk with exponential backoff + jitter on 429 / network errors.

    status_label is only used for clearer progress messages, for example:
    "site 101 | chunk 2/3".
    """
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "start_date": start,
        "end_date":   end,
        "hourly":     ",".join(HOURLY_VARIABLES),
        "timezone":   "Europe/Brussels",
    }

    prefix = f"[{status_label}] " if status_label else ""

    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params, timeout=60)

            if resp.status_code == 429:
                wait = min(300, 60 * (2 ** attempt)) + random.uniform(0, 15)
                print(f"\n    {prefix}[429] rate limited — waiting {wait:.0f}s "
                      f"(attempt {attempt+1}/{max_retries})...",
                      end=" ", flush=True)
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()

        except requests.exceptions.ConnectionError:
            wait = min(120, 30 * (attempt + 1)) + random.uniform(0, 10)
            print(f"\n    {prefix}[ConnectionError] waiting {wait:.0f}s "
                  f"(attempt {attempt+1}/{max_retries})...",
                  end=" ", flush=True)
            time.sleep(wait)

        except requests.exceptions.Timeout:
            wait = 30 + random.uniform(0, 10)
            print(f"\n    {prefix}[Timeout] waiting {wait:.0f}s "
                  f"(attempt {attempt+1}/{max_retries})...",
                  end=" ", flush=True)
            time.sleep(wait)

    raise RuntimeError(f"Failed after {max_retries} retries ({start} → {end})")


# Crawl 
all_rows, complete_sites = load_existing(OUTPUT_FILE)
failed_chunks = []   # track chunks that failed even after retries
print()

TOTAL_SITES = len(SITES)
TOTAL_CHUNKS = len(chunks)
start_time = datetime.now()

print(f"Starting crawl at {start_time:%Y-%m-%d %H:%M:%S}")
print(f"Sites to check: {TOTAL_SITES} | already complete: {len(complete_sites)} | chunks/site: {TOTAL_CHUNKS}\n")

for ri, site_info in enumerate(SITES, start=1):
    site_id = site_info["site_id"]
    lat     = float(site_info["latitude"])
    lon     = float(site_info["longitude"])

    if site_id in complete_sites:
        print(f"[SITE {ri:>3}/{TOTAL_SITES}] site_id={site_id} — SKIPPED (already complete)")
        continue

    all_rows = [r for r in all_rows if int(r["site_id"]) != int(site_id)]
    print("=" * 80)
    print(f"[SITE {ri:>3}/{TOTAL_SITES}] NOW CRAWLING site_id={site_id} | lat={lat:.5f}, lon={lon:.5f}")
    site_records = 0

    for chunk_i, (api_type, chunk_start, chunk_end) in enumerate(chunks, start=1):
        url       = ARCHIVE_URL if api_type == "archive" else FORECAST_URL
        start_str = chunk_start.strftime("%Y-%m-%d")
        end_str   = chunk_end.strftime("%Y-%m-%d")
        status_label = f"site {ri}/{TOTAL_SITES} id={site_id} | chunk {chunk_i}/{TOTAL_CHUNKS}"

        print(
            f"    [{status_label}] {api_type:>8} {start_str} → {end_str} ...",
            end=" ",
            flush=True,
        )

        try:
            data   = fetch_weather(url, lat, lon, start_str, end_str, status_label=status_label)
            hourly = data.get("hourly", {})
            times  = hourly.get("time", [])

            for j, timestamp in enumerate(times):
                row = {"site_id": site_id, "latitude": lat,
                       "longitude": lon, "datetime": timestamp}
                for var in HOURLY_VARIABLES:
                    values   = hourly.get(var, [])
                    row[var] = values[j] if j < len(values) else None
                all_rows.append(row)

            site_records += len(times)
            print(f"OK — {len(times):,} records")

        except Exception as e:
            print(f"FAILED: {e}")
            failed_chunks.append({"site_id": site_id, "chunk": f"{start_str}→{end_str}", "error": str(e)})

        time.sleep(DELAY_BETWEEN_CHUNKS)   # always wait between chunks

    elapsed = datetime.now() - start_time
    print(f"[SITE {ri:>3}/{TOTAL_SITES}] FINISHED site_id={site_id} — {site_records:,} records | elapsed: {elapsed}")
    print(f"Sleeping {DELAY_BETWEEN_SITES}s before next site...\n")
    time.sleep(DELAY_BETWEEN_SITES)        # extra pause between sites

    # Save incrementally every 10 sites — don't lose progress on crash
    if ri % 10 == 0:
        fieldnames = ["site_id", "latitude", "longitude", "datetime"] + HOURLY_VARIABLES
        with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_rows)
        print(f"[CHECKPOINT] saved {len(all_rows):,} rows after site {ri}/{TOTAL_SITES} (site_id={site_id})\n")

print(f"\nCrawl complete. {len(all_rows):,} total rows in memory.")
if failed_chunks:
    print(f"\n{len(failed_chunks)} chunks failed:")
    for fc in failed_chunks:
        print(f"  site {fc['site_id']}  {fc['chunk']}  — {fc['error']}")



No existing file found — starting fresh.

[  1/150] site 2 (51.27512, 4.47169)
    [ archive] 2019-08-01 → 2019-12-31 ... 3,672 records
    [ archive] 2020-01-01 → 2020-12-31 ... 8,784 records
    [ archive] 2021-01-01 → 2021-12-31 ... 8,760 records
    [ archive] 2022-01-01 → 2022-12-31 ... 8,760 records
    [ archive] 2023-01-01 → 2023-12-31 ... 8,760 records
    [ archive] 2024-01-01 → 2024-12-31 ... 8,784 records
    [ archive] 2025-01-01 → 2025-12-31 ... 8,760 records
    [ archive] 2026-01-01 → 2026-03-15 ... 1,776 records
    [forecast] 2026-03-15 → 2026-03-25 ... 264 records
    → 58,320 records | sleeping 8.0s

[  2/150] site 3 (51.27503, 4.47222)
    [ archive] 2019-08-01 → 2019-12-31 ... 3,672 records
    [ archive] 2020-01-01 → 2020-12-31 ... 8,784 records
    [ archive] 2021-01-01 → 2021-12-31 ... 8,760 records
    [ archive] 2022-01-01 → 2022-12-31 ... 8,760 records
    [ archive] 2023-01-01 → 2023-12-31 ... 8,760 records
    [ archive] 2024-01-01 → 2024-12-31 ... 8,784 r

## Save to CSV

In [ ]:
fieldnames = ["site_id", "latitude", "longitude", "datetime"] + HOURLY_VARIABLES

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)

print(f"Saved {len(all_rows):,} rows → {OUTPUT_FILE}")


## Preview output

In [ ]:
df_weather = pd.read_csv(OUTPUT_FILE)
print(df_weather.shape)
print("\nRecords per site:")
print(df_weather["site_id"].value_counts().to_string())
print()
df_weather.head()
